# air-gap DB clone — profiler + generator

Инструмент «безопасного клонирования» БД через воздушный зазор из **двух отдельных программ**. Через зазор едет **не выгрузка данных, а маленький человекочитаемый `profile.json`** (схема + статистика) — его можно глазами проверить перед переносом.

```
закрытый контур (реальная БД)                 открытый контур (LLM, без реальной БД)
┌───────────────────────────┐   profile.json  ┌────────────────────────────────────┐
│  profiler                 │  ─────────────▶ │  generator                         │
│  каталог + pg_stats        │   (+ policy)    │  schema.sql + синтетические данные  │
│  → profile.json (аудит)    │                 │  (CSV или INSERT-ы)                │
└───────────────────────────┘                 └────────────────────────────────────┘
```

Этот ноутбук:
1. пересоздаёт весь инструмент в папке **`./generated`** (ячейки `%%writefile`);
2. запускает **profiler** на подключении БД проекта (`config_db.json`/`.env`); если БД недоступна — берёт готовый пример профиля;
3. запускает **generator** → `schema.sql` + синтетические данные в **`./generated/output`**.

> Все файлы генерируются в папке `generated/` рядом с этим ноутбуком.

## 1. Подключение к БД, входные данные и настройки

Подключение задаётся **прямо здесь** (модуль самостоятельный — из проекта ничего не берём). Список таблиц — **через запятую** `schema.table` (или CSV-файлом).

> Greenplum на проде часто ходит по Kerberos/GSSAPI — тогда пароль не указывайте, драйвер возьмёт тикет из окружения.

In [ ]:
# ── Подключение к БД (закрытый контур) ─────────────────────────────────────
# Вариант A: готовый DSN одной строкой.
DB_DSN = "postgresql://user@host:5432/prom"   # без пароля → Kerberos/GSSAPI
# Вариант B: собрать из компонентов (если DB_DSN пуст).
DB_HOST, DB_PORT, DB_NAME = "host", 5432, "prom"
DB_USER, DB_PASSWORD = "user", ""             # пароль пустой → Kerberos/ident

# ── Входные данные ─────────────────────────────────────────────────────────
# Таблицы для обработки: schema.table через запятую.
TABLES = "public.clients,public.tasks"
# Либо путь к CSV со списком таблиц (колонки schema,table). Если задан — приоритетнее.
TABLES_CSV = ""            # напр. "generated/tables.example.csv"

# ── Настройки ──────────────────────────────────────────────────────────────
POLICY_YAML   = "generated/policy.example.yaml"   # whitelist + зависимости (или "")
SAMPLE_N      = 1_000_000  # сколько строк тянуть в сэмпл на таблицу (pandas в памяти)
SCALE_FACTOR  = 0.001      # доля от исходного числа строк (1.2M × 0.001 ≈ 1200)
SEED          = 42         # детерминизм генератора
OUTPUT_FORMAT = "csv"      # "csv" (файл на таблицу) | "sql" (батч INSERT-ов)
KEEP_GPDB     = False      # сохранять DISTRIBUTED BY / партиции в DDL

from pathlib import Path
NB_DIR = Path.cwd()                 # папка с этим ноутбуком (air_gap_clone/)
GEN = NB_DIR / "generated"          # сюда генерируются ВСЕ файлы инструмента
OUT = GEN / "output"                # артефакты запуска (profile.json, schema.sql, данные)
OUT.mkdir(parents=True, exist_ok=True)
print("Файлы инструмента →", GEN)
print("Артефакты запуска →", OUT)

## 2. Генерация файлов инструмента в `./generated`

Ячейки ниже (`%%writefile`) записывают модули обеих программ на диск. Это и есть готовый к переносу инструмент — его можно запускать и отдельно от ноутбука (`python generated/profiler.py ...`, `python generated/generator.py ...`).

In [ ]:
# Каталоги инструмента.
for d in ["generated", "generated/agc_profiler", "generated/agc_generator", "generated/examples"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Каталоги готовы")

### Общий модуль

`agc_common.py` — валидация SQL-идентификаторов и логирование, используется обеими программами.

In [ ]:
%%writefile generated/agc_common.py
"""Общие утилиты обеих программ: валидация идентификаторов, логирование.

Никакой бизнес-логики — только то, что нужно и профайлеру, и генератору.
"""
from __future__ import annotations

import hashlib
import logging
import re
import sys


def stable_hash(*parts) -> int:
    """Детерминированный хэш строк в int — НЕ зависит от PYTHONHASHSEED (в отличие от
    builtin hash()). Нужен, чтобы --seed давал одинаковый результат между запусками."""
    s = "\x1f".join(str(p) for p in parts)
    return int.from_bytes(hashlib.blake2b(s.encode("utf-8"), digest_size=8).digest(), "big")

# SQL-идентификатор: буква/underscore, далее буквы/цифры/underscore/$.
# Всё, что не проходит, — не подставляем в SQL как имя объекта (защита от инъекции).
_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")


def validate_identifier(name: str, kind: str = "identifier") -> str:
    """Проверяет, что name — безопасный SQL-идентификатор. Иначе ValueError.

    Значения в запросы всегда идут параметрами; идентификаторы (schema/table/
    column) параметризовать нельзя, поэтому валидируем их этой функцией.
    """
    if not isinstance(name, str) or not _IDENT_RE.match(name):
        raise ValueError(f"Недопустимый SQL-{kind}: {name!r}")
    return name


def quote_ident(name: str) -> str:
    """Экранирует идентификатор двойными кавычками (для DDL/SQL)."""
    return '"' + name.replace('"', '""') + '"'


def qualified(schema: str, table: str) -> str:
    return f"{quote_ident(schema)}.{quote_ident(table)}"


def get_logger(name: str = "agc") -> logging.Logger:
    """Единый логгер в stderr. Пишем: что прочитано, что засинтезировано, warning-и."""
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stderr)
        handler.setFormatter(
            logging.Formatter("%(asctime)s %(levelname)-7s [%(name)s] %(message)s", "%H:%M:%S")
        )
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger


### Программа 1 — profiler (закрытый контур)

Модули: `db` (подключение, read-only, самостоятельное) · `catalog_reader` (структура из каталога) · `sampler` (сэмпл в pandas) · `analyze` (PK-гипотеза, статистика, категории, зависимости) · `classifier` (эвристики-предложения) · `policy` (whitelist + зависимости) · `linter` (аудит утечек) · `profile_builder` (сборка) · `cli`.

In [ ]:
%%writefile generated/agc_profiler/__init__.py
"""Программа 1 — profiler (закрытый контур).

Читает схему и статистику реальной БД из системного каталога и pg_stats,
применяет whitelist-политику чувствительности и пишет компактный profile.json.
Реальные строки данных наружу не выгружаются (см. linter — граница аудита).
"""


In [ ]:
%%writefile generated/agc_profiler/db.py
"""Подключение к БД закрытого контура (единственный сетевой вызов инструмента).

САМОСТОЯТЕЛЬНЫЙ модуль — ничего из внешних проектов не импортируем. Параметры
подключения задаются явно (в ячейке ноутбука, через --dsn или db_config.json).

Порядок разрешения DSN:
  1) явный dsn (аргумент / --dsn / ячейка ноутбука);
  2) переменные окружения AGC_DB_DSN, затем DB_DSN;
  3) db_config.json рядом (ключ "dsn" ИЛИ поля host/port/database/user[/password]).

Движок — read-only: default_transaction_read_only=on + statement_timeout,
чтобы профайлер физически не мог ничего записать в реальную БД.

Greenplum на проде часто ходит по Kerberos/GSSAPI (без пароля) — тогда просто не
указывайте password, драйвер возьмёт тикет из окружения.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from sqlalchemy import create_engine
from sqlalchemy.engine import Engine

from agc_common import get_logger

log = get_logger("profiler.db")


def dsn_from_parts(host: str, database: str, *, port: int = 5432,
                   user: str = "", password: str = "", driver: str = "postgresql") -> str:
    """Собрать DSN из компонентов. password пустой → Kerberos/ident (как на проде GPDB)."""
    auth = user
    if user and password:
        auth = f"{user}:{password}"
    at = f"{auth}@" if auth else ""
    return f"{driver}://{at}{host}:{port}/{database}"


def resolve_dsn(dsn: str | None = None, config_path: str | Path | None = None) -> str:
    if dsn:
        return dsn
    for env in ("AGC_DB_DSN", "DB_DSN"):
        if os.getenv(env):
            return os.environ[env]
    if config_path and Path(config_path).exists():
        data = json.loads(Path(config_path).read_text(encoding="utf-8"))
        if data.get("dsn"):
            return str(data["dsn"])
        return dsn_from_parts(
            data["host"], data["database"],
            port=int(data.get("port", 5432)),
            user=data.get("user") or data.get("user_id") or "",
            password=data.get("password", ""),
            driver=data.get("driver", "postgresql"),
        )
    raise RuntimeError(
        "Не задан DSN БД. Укажите dsn/--dsn, переменную AGC_DB_DSN/DB_DSN или db_config.json "
        "(host/port/database/user[/password])."
    )


def make_engine(
    dsn: str | None = None,
    config_path: str | Path | None = None,
    statement_timeout_ms: int = 600_000,
) -> Engine:
    """Read-only SQLAlchemy-движок к Greenplum/Postgres (драйвер psycopg2)."""
    url = resolve_dsn(dsn, config_path)
    safe = url.split("@", 1)[-1] if "@" in url else url  # креды в лог не пишем
    log.info("Подключение к БД (read-only): ...@%s", safe)
    opts = (
        f"-c statement_timeout={int(statement_timeout_ms)} "
        f"-c default_transaction_read_only=on"
    )
    return create_engine(url, pool_pre_ping=True, connect_args={"options": opts})


In [ ]:
%%writefile generated/agc_profiler/catalog_reader.py
"""Чтение СТРУКТУРЫ из системного каталога — это ground truth, дёшево и без скана.

Читаем: типы/nullability/default, PK/UNIQUE/NOT NULL/FK/CHECK, ключ распределения
Greenplum (gp_distribution_policy), партиционирование, тип хранения (heap /
append-optimized / column-oriented), relkind (таблица/вью).

Версия GPDB заранее неизвестна — GPDB-специфику (gp_distribution_policy,
pg_partitions, pg_appendonly) оборачиваем в try/except и мягко деградируем.
"""
from __future__ import annotations

import re

from sqlalchemy import text
from sqlalchemy.engine import Engine

from agc_common import get_logger, validate_identifier

log = get_logger("profiler.catalog")

_NUMTYPE_RE = re.compile(r"^(?:numeric|decimal)\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.IGNORECASE)
_CHARLEN_RE = re.compile(r"\(\s*(\d+)\s*\)")

_RELKIND = {
    "r": "table", "v": "view", "m": "matview",
    "f": "foreign_table", "p": "partitioned_table", "t": "toast",
}


def read_table_meta(engine: Engine, schema: str, table: str) -> dict:
    """relkind, оценка reltuples, oid, тип хранения."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = text(
        "SELECT c.oid AS oid, c.relkind AS relkind, c.reltuples::bigint AS reltuples "
        "FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace "
        "WHERE n.nspname = :s AND c.relname = :t"
    )
    with engine.connect() as conn:
        row = conn.execute(sql, {"s": schema, "t": table}).mappings().first()
    if row is None:
        raise LookupError(f"Объект {schema}.{table} не найден в pg_class")
    oid = int(row["oid"])
    reltuples = int(row["reltuples"] or 0)
    meta = {
        "relkind": _RELKIND.get(row["relkind"], row["relkind"]),
        "is_view": row["relkind"] in ("v", "m"),
        "reltuples": reltuples,
        # reltuples всегда лишь оценка (обновляется ANALYZE); 0 — почти наверняка stale.
        "row_count_estimated": True,
        "storage": _read_storage(engine, oid),
        "oid": oid,
    }
    return meta


def _read_storage(engine: Engine, oid: int) -> str:
    """heap / append_optimized_row / append_optimized_column. GPDB-специфика — guarded."""
    try:
        with engine.connect() as conn:
            row = conn.execute(
                text("SELECT columnstore FROM pg_appendonly WHERE relid = :oid"),
                {"oid": oid},
            ).mappings().first()
        if row is None:
            return "heap"
        return "append_optimized_column" if row["columnstore"] else "append_optimized_row"
    except Exception:  # noqa: BLE001 — не GPDB / нет pg_appendonly
        return "heap"


def read_columns(engine: Engine, schema: str, table: str) -> list[dict]:
    """Колонки: имя, точный pg_type (format_type), nullability, default, precision/scale."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = text(
        "SELECT a.attnum AS attnum, a.attname AS name, "
        "       format_type(a.atttypid, a.atttypmod) AS pg_type, "
        "       a.attnotnull AS notnull, "
        "       pg_get_expr(ad.adbin, ad.adrelid) AS default_expr "
        "FROM pg_attribute a "
        "JOIN pg_class c ON c.oid = a.attrelid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "LEFT JOIN pg_attrdef ad ON ad.adrelid = a.attrelid AND ad.adnum = a.attnum "
        "WHERE n.nspname = :s AND c.relname = :t AND a.attnum > 0 AND NOT a.attisdropped "
        "ORDER BY a.attnum"
    )
    cols: list[dict] = []
    with engine.connect() as conn:
        for row in conn.execute(sql, {"s": schema, "t": table}).mappings():
            pg_type = str(row["pg_type"])
            precision, scale = _parse_numeric(pg_type)
            cols.append({
                "name": str(row["name"]),
                "pg_type": pg_type,
                "nullable": not bool(row["notnull"]),
                "default": row["default_expr"],
                "precision": precision,
                "scale": scale,
                "char_len": _parse_charlen(pg_type),
                "ordinal": int(row["attnum"]),
            })
    return cols


def _parse_numeric(pg_type: str) -> tuple[int | None, int | None]:
    m = _NUMTYPE_RE.match(pg_type)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)


def _parse_charlen(pg_type: str) -> int | None:
    if pg_type.lower().startswith(("character", "varchar", "char")):
        m = _CHARLEN_RE.search(pg_type)
        return int(m.group(1)) if m else None
    return None


# NOTE: PK/FK из DDL не читаем — в целевых таблицах они не объявлены. PK выводим
# гипотезой по сэмплу (analyze.find_pk, как в исходном проекте); FK не выводим
# вовсе (ключи джойнов подбираются позже на синтетике). CHECK/UNIQUE тоже опускаем.


def read_distribution(engine: Engine, schema: str, table: str) -> list[str]:
    """Ключ распределения Greenplum из gp_distribution_policy. [] если не GPDB
    или распределение случайное/реплицированное."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    # GPDB 6+: колонка distkey (int2vector). GPDB 5: attrnums (smallint[]).
    variants = (
        "SELECT att.attname AS col "
        "FROM gp_distribution_policy p "
        "JOIN pg_class c ON c.oid = p.localoid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "JOIN unnest(p.distkey) WITH ORDINALITY k(a, o) ON true "
        "JOIN pg_attribute att ON att.attrelid = c.oid AND att.attnum = k.a "
        "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o",
        "SELECT att.attname AS col "
        "FROM gp_distribution_policy p "
        "JOIN pg_class c ON c.oid = p.localoid "
        "JOIN pg_namespace n ON n.oid = c.relnamespace "
        "JOIN unnest(p.attrnums) WITH ORDINALITY k(a, o) ON true "
        "JOIN pg_attribute att ON att.attrelid = c.oid AND att.attnum = k.a "
        "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o",
    )
    for sql in variants:
        try:
            with engine.connect() as conn:
                rows = conn.execute(text(sql), {"s": schema, "t": table}).scalars().all()
            return [str(r) for r in rows]
        except Exception:  # noqa: BLE001 — пробуем следующий вариант / не GPDB
            continue
    return []


def read_partition_keys(engine: Engine, schema: str, table: str) -> list[str]:
    """Колонки партиционирования. GPDB classic (pg_partition_columns) → PG native
    (pg_partitioned_table). [] если не партиционировано / представление недоступно."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    # GPDB classic partitioning.
    try:
        with engine.connect() as conn:
            rows = conn.execute(text(
                "SELECT columnname FROM pg_partition_columns "
                "WHERE schemaname = :s AND tablename = :t "
                "ORDER BY position_in_partition_key"
            ), {"s": schema, "t": table}).scalars().all()
        if rows:
            return [str(r) for r in rows]
    except Exception:  # noqa: BLE001
        pass
    # PG native declarative partitioning (GPDB 7 / Postgres).
    try:
        with engine.connect() as conn:
            cols = conn.execute(text(
                "SELECT a.attname "
                "FROM pg_partitioned_table pt "
                "JOIN pg_class c ON c.oid = pt.partrelid "
                "JOIN pg_namespace n ON n.oid = c.relnamespace "
                "JOIN unnest(pt.partattrs) WITH ORDINALITY k(a, o) ON true "
                "JOIN pg_attribute a ON a.attrelid = c.oid AND a.attnum = k.a "
                "WHERE n.nspname = :s AND c.relname = :t ORDER BY k.o"
            ), {"s": schema, "t": table}).scalars().all()
        return [str(c) for c in cols]
    except Exception:  # noqa: BLE001
        return []


In [ ]:
%%writefile generated/agc_profiler/sampler.py
"""Адаптивный сэмплер: тянет случайную выборку строк таблицы в pandas.

Вся статистика профиля считается в pandas НА ЭТОМ СЭМПЛЕ (как в исходном проекте),
а не отдельными GROUP BY к БД. На контуре с большим RAM это дёшево и быстро.

Оценка числа строк est берётся из планировщика (EXPLAIN, без выполнения) или из
reltuples. Стратегия:
  - 0 < est <= порога → ORDER BY random() LIMIT :n  (точная выборка, дёшево на малых);
  - est > порога      → WHERE random() < p LIMIT :n, p=min(1, oversample*n/est) (без сортировки);
  - est <= 0 / None    → считаем объект большим (assumed rows), берём p-стратегию.

ОГОВОРКА (смещение): `WHERE random() < p LIMIT n` обрывает скан по достижении n
строк — это «начало физического скана», а не равномерная выборка. Для GPDB
(партиции, append-optimized, порядок вставки) это даёт смещение по содержимому;
редкие категории в сэмпле могут потеряться (это допустимо по договорённости).
"""
from __future__ import annotations

import json

import pandas as pd
from sqlalchemy import text
from sqlalchemy.engine import Engine

from agc_common import get_logger, validate_identifier

log = get_logger("profiler.sampler")

SAMPLE_SORT_THRESHOLD = 2_000_000
SAMPLE_OVERSAMPLE = 2.0
SAMPLE_ASSUMED_ROWS_UNKNOWN = 10_000_000


def estimate_row_count(engine: Engine, schema: str, table: str) -> int | None:
    """Оценка числа строк через планировщик (EXPLAIN FORMAT JSON, без скана).
    Работает и для вью. None — оценить не удалось."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    sql = f'EXPLAIN (FORMAT JSON) SELECT * FROM "{schema}"."{table}"'
    try:
        with engine.connect() as conn:
            raw = conn.execute(text(sql)).scalar()
        data = json.loads(raw) if isinstance(raw, str) else raw
        rows = int(data[0]["Plan"]["Plan Rows"])
        return rows if rows > 0 else None
    except Exception as exc:  # noqa: BLE001
        log.info("estimate_row_count(%s.%s) не удалась: %s", schema, table, exc)
        return None


def build_sample_sql(schema: str, table: str, n: int, est: int | None) -> str:
    """SQL адаптивного сэмпла `SELECT *` по оценке est (см. оговорку в docstring модуля)."""
    validate_identifier(schema, "schema")
    validate_identifier(table, "table")
    base = f'SELECT * FROM "{schema}"."{table}"'
    if est is not None and 0 < est <= SAMPLE_SORT_THRESHOLD:
        return base + " ORDER BY random() LIMIT :n"
    denom = est if (est and est > 0) else SAMPLE_ASSUMED_ROWS_UNKNOWN
    p = min(1.0, SAMPLE_OVERSAMPLE * float(n) / float(denom))
    return base + f" WHERE random() < {p:.10g} LIMIT :n"


def sample_dataframe(engine: Engine, schema: str, table: str, n: int = 1_000_000,
                     *, est: int | None = None, timeout_ms: int | None = None) -> pd.DataFrame:
    """Случайный сэмпл до n строк таблицы целиком → pandas.DataFrame."""
    if est is None:
        est = estimate_row_count(engine, schema, table)
    sql = build_sample_sql(schema, table, n, est)
    strategy = "sort_random" if "ORDER BY random()" in sql else "random_filter"
    log.info("sample %s.%s: strategy=%s est=%s n=%d", schema, table, strategy, est, n)
    with engine.connect() as conn:
        if timeout_ms:
            conn.execute(text(f"SET statement_timeout = {int(timeout_ms)}"))
        res = conn.execute(text(sql), {"n": int(n)})
        cols = list(res.keys())
        rows = res.fetchall()
    df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)
    log.info("sample %s.%s: получено %d строк, %d колонок", schema, table, len(df), len(cols))
    return df


In [ ]:
%%writefile generated/agc_profiler/analyze.py
"""Анализ сэмпла в pandas: PK-гипотеза, статистика колонок, категории,
представители функциональных зависимостей.

PK нигде не объявлен в DDL — выводим гипотезу по сэмплу (минимальная уникальная
комбинация), как в исходном проекте. FK НЕ выводим: связи джойнов подбираются
позже на синтетике.
"""
from __future__ import annotations

import re
from itertools import combinations

import pandas as pd

from agc_common import get_logger

log = get_logger("profiler.analyze")

# Категория = меньше этого числа уникальных значений в сэмпле.
CATEGORICAL_MAX_DISTINCT = 200

_METRIC_RE = re.compile(
    r"(^|_)(qty|quantity|amt|amount|sum|total|cnt|count|avg|rate|ratio|pct|perc|percent|val|value)($|_)",
    re.I)
# Системные таймстемпы почти уникальны, но НЕ бизнес-ключи — откладываем в PK-поиске.
_SYS_TS_RE = re.compile(r"(dttm$|timestamp|inserted|modified|updated|_update_|^update_|load_)", re.I)


def _is_metric_name(name: str) -> bool:
    return bool(_METRIC_RE.search(name or ""))


def find_pk(df: pd.DataFrame, max_cols: int = 4) -> list[str]:
    """Минимальная уникальная комбинация на сэмпле (PK-гипотеза).

    Бизнес-ключи в приоритете: метрики и системные таймстемпы откладываем — иначе
    почти-уникальный load-таймстемп ложно становится PK. Возвращает [] если не нашли.
    ВНИМАНИЕ: это гипотеза по сэмплу — уникальность в сэмпле не доказывает PK.
    """
    if df.empty:
        return []
    cols = [c for c in df.columns if df[c].notna().all() and df[c].nunique(dropna=False) > 1]
    if not cols:
        return []

    def _low_priority(c: str) -> bool:
        return _is_metric_name(c) or bool(_SYS_TS_RE.search(c))

    preferred = [c for c in cols if not _low_priority(c)]
    deferred = [c for c in cols if _low_priority(c)]
    for candidates in ([preferred] if preferred else []) + [preferred + deferred]:
        upper = min(max_cols, len(candidates))
        for size in range(1, upper + 1):
            for combo in combinations(candidates, size):
                if not df.duplicated(subset=list(combo)).any():
                    return list(combo)
    return []


def column_stats(df: pd.DataFrame, col: str) -> dict:
    """not_null_perc, null_frac, n_distinct (в сэмпле), unique_perc."""
    n = len(df)
    if n == 0 or col not in df.columns:
        return {"not_null_perc": 0.0, "null_frac": 0.0, "n_distinct": 0, "unique_perc": 0.0}
    notna = int(df[col].notna().sum())
    n_distinct = int(df[col].dropna().nunique())
    return {
        "not_null_perc": round(notna / n * 100, 2),
        "null_frac": round(1 - notna / n, 6),
        "n_distinct": n_distinct,
        "unique_perc": round(n_distinct / n * 100, 2),
    }


def top_categories(df: pd.DataFrame, col: str, cap: int = CATEGORICAL_MAX_DISTINCT) -> list[list]:
    """Все категории в сэмпле c долями: [[value, freq], ...] по убыванию частоты.

    Доли считаются от НЕ-NULL значений (NULL-долю несёт отдельный null_frac).
    Редкие категории, не попавшие в сэмпл, теряются — это допустимо.
    """
    if col not in df.columns:
        return []
    vc = df[col].dropna().value_counts(normalize=True)
    out = [[_norm(v), round(float(f), 6)] for v, f in vc.head(cap).items()]
    return out


def dependency_representatives(df: pd.DataFrame, determinant: str, dependent: str,
                               order_by: str | None = None) -> dict[str, str]:
    """Для каждой категории determinant — ОДИН представитель dependent.

    Берём строку с самым большим order_by (если задан), где dependent не NULL.
    Так «опросник» остаётся привязан к своему подтипу задачи. Если order_by нет —
    берём первое встретившееся не-NULL значение.
    """
    if determinant not in df.columns or dependent not in df.columns:
        return {}
    sub = df[df[dependent].notna()]
    if sub.empty:
        return {}
    if order_by and order_by in sub.columns:
        sub = sub.sort_values(order_by, ascending=True, na_position="first")
        picked = sub.groupby(determinant, dropna=True)[dependent].last()
    else:
        picked = sub.groupby(determinant, dropna=True)[dependent].first()
    return {_norm(k): _norm(v) for k, v in picked.items()}


def _norm(v):
    """Значение → JSON-безопасный скаляр (строка/число/None)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, (int, float, bool)):
        return v
    return str(v)


In [ ]:
%%writefile generated/agc_profiler/classifier.py
"""Авто-эвристики классификации колонок — они только ПРЕДЛАГАЮТ класс.

Финальное решение принимает policy (см. policy.py): всё, что не одобрено явно,
синтезируется. Классификатор не может пометить колонку categorical_keep — это
делает только whitelist в policy-файле (граница безопасности).

Классы: categorical_candidate, sensitive_numeric, pii, key, datetime, sensitive.
Для pii дополнительно предлагаем имя генератора (full_name/email/phone/...).
"""
from __future__ import annotations

import re

# Регэкспы форматов значений (проверяются по сэмплу коротких полей).
_RE_EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
_RE_PHONE = re.compile(r"^[+()\-\s\d]{7,20}$")
_RE_INN = re.compile(r"^\d{10}$|^\d{12}$")
_RE_SNILS = re.compile(r"^\d{3}-\d{3}-\d{3} \d{2}$|^\d{11}$")
_RE_ACCOUNT = re.compile(r"^\d{20}$")

# Подсказки по ИМЕНИ колонки → (класс, генератор).
_NAME_HINTS = (
    (re.compile(r"e?mail", re.I), "pii", "email"),
    (re.compile(r"phone|tel|mobile|телефон", re.I), "pii", "phone"),
    (re.compile(r"\binn\b|_inn|инн", re.I), "pii", "inn"),
    (re.compile(r"snils|снилс", re.I), "pii", "snils"),
    (re.compile(r"account|schet|счет|счёт|\bacc\b|iban|card", re.I), "pii", "account"),
    (re.compile(r"passport|паспорт", re.I), "pii", "passport"),
    (re.compile(r"fio|full_?name|fam|surname|lastname|firstname|имя|фамилия|фио", re.I), "pii", "full_name"),
    (re.compile(r"\bname\b|_name$|наимен", re.I), "pii", "full_name"),
    (re.compile(r"address|addr|адрес", re.I), "pii", "address"),
    (re.compile(r"company|org|orgname|организац|компан", re.I), "pii", "company"),
    (re.compile(r"birth|dob|рожд", re.I), "datetime", None),
)

_NUMERIC_TYPES = ("smallint", "integer", "bigint", "numeric", "decimal",
                  "real", "double precision", "money")
_DATETIME_TYPES = ("date", "timestamp", "time")
# Категория = меньше этого числа уникальных значений в сэмпле.
CATEGORICAL_MAX_DISTINCT = 200


def _base_type(pg_type: str) -> str:
    return (pg_type or "").split("(")[0].strip().lower()


def _looks_like(values: list, regex: re.Pattern) -> bool:
    """>=80% непустых сэмпл-значений матчат regex."""
    vals = [str(v) for v in values if v is not None and str(v) != ""]
    if len(vals) < 3:
        return False
    hits = sum(1 for v in vals if regex.match(v))
    return hits / len(vals) >= 0.8


def propose(
    name: str,
    pg_type: str,
    *,
    is_pk: bool = False,
    n_distinct: float | None = None,
    sample_values: list | None = None,
) -> tuple[str, str | None]:
    """Возвращает (предложенный_класс, генератор|None). FK не выводим — только PK."""
    base = _base_type(pg_type)

    if is_pk:
        return "key", None

    # Формат значений из сэмпла (только короткие поля попадают сюда со значениями).
    sv = sample_values or []
    if _looks_like(sv, _RE_EMAIL):
        return "pii", "email"
    if _looks_like(sv, _RE_ACCOUNT):
        return "pii", "account"
    if _looks_like(sv, _RE_INN):
        return "pii", "inn"
    if _looks_like(sv, _RE_SNILS):
        return "pii", "snils"

    # Подсказки по имени.
    for regex, klass, gen in _NAME_HINTS:
        if regex.search(name):
            return klass, gen

    if base in _DATETIME_TYPES:
        return "datetime", None

    # Текст с низкой кардинальностью → КАНДИДАТ в категориальные (не keep!).
    if base in ("text", "character varying", "varchar", "character", "char"):
        if n_distinct is not None and 0 < n_distinct <= CATEGORICAL_MAX_DISTINCT:
            return "categorical_candidate", None
        return "sensitive", "text"

    if base in _NUMERIC_TYPES:
        # Маленькая кардинальность у числа — тоже кандидат в категориальные (коды/флаги).
        if n_distinct is not None and 0 < n_distinct <= CATEGORICAL_MAX_DISTINCT:
            return "categorical_candidate", None
        return "sensitive_numeric", None

    if base in ("boolean",):
        return "categorical_candidate", None

    return "sensitive", "text"


In [ ]:
%%writefile generated/agc_profiler/policy.py
"""Policy — whitelist чувствительности (граница безопасности).

По умолчанию КАЖДАЯ колонка считается чувствительной и будет синтезироваться.
Реальные значения (most_common_vals/histogram_bounds) попадают в профиль ТОЛЬКО
для колонок, явно помеченных в policy как categorical_keep. Никакая авто-эвристика
не может выдать categorical_keep — это делает только человек в YAML.

Формат policy-файла (YAML):

    version: 1
    columns:
      "schema.table.col":  categorical_keep          # короткая форма
      "schema.table.name": {class: pii, generator: full_name}
      "schema.table.amt":  {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}
    tables:
      "schema.table":
        order_groups: [["created_at", "updated_at"]]  # даты: created <= updated
        dependencies:                                  # функциональные зависимости
          - determinant: task_subtype                  # категория-детерминант
            dependent:   task_questionary               # зависимая колонка
            order_by:    task_date                      # (опц.) представитель — свежайший
            keep_representative: true                   # хранить реальный представитель (whitelist)

Разрешённые классы в policy: categorical_keep, sensitive_numeric, pii, key,
datetime, sensitive.

Зависимость: для каждой категории determinant берётся ОДИН представитель dependent
(строка с максимальным order_by, где dependent не NULL). В синтетике та же категория
получает тот же представитель — «опросник остаётся у своего подтипа». При
keep_representative=true реальный представитель (whitelist-opt-in) кладётся в профиль;
иначе генератор ставит стабильный синтетический токен на категорию.
"""
from __future__ import annotations

from pathlib import Path

import yaml

from agc_common import get_logger

log = get_logger("profiler.policy")

VALID_CLASSES = {
    "categorical_keep", "sensitive_numeric", "pii", "key", "datetime", "sensitive",
}
# Классы, которые НЕЛЬЗЯ вывести автоматически — только явным whitelist.
WHITELIST_ONLY = {"categorical_keep"}


class Policy:
    def __init__(self, columns: dict, tables: dict):
        self._columns = columns          # "s.t.c" -> {"class":..., extras...}
        self._tables = tables            # "s.t"   -> {"order_groups": [...]}

    @classmethod
    def load(cls, path: str | Path | None) -> "Policy":
        if not path:
            log.info("Policy-файл не задан — все колонки трактуются как чувствительные.")
            return cls({}, {})
        data = yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}
        columns: dict[str, dict] = {}
        for key, val in (data.get("columns") or {}).items():
            if isinstance(val, str):
                entry = {"class": val}
            elif isinstance(val, dict):
                entry = dict(val)
            else:
                raise ValueError(f"policy.columns[{key!r}] должен быть строкой или dict")
            klass = entry.get("class")
            if klass not in VALID_CLASSES:
                raise ValueError(f"policy.columns[{key!r}].class={klass!r} не из {VALID_CLASSES}")
            columns[key] = entry
        tables = dict(data.get("tables") or {})
        log.info("Policy загружена: %d правил по колонкам, %d по таблицам.",
                 len(columns), len(tables))
        return cls(columns, tables)

    def resolve(self, schema: str, table: str, column: str, proposed: str,
                proposed_gen: str | None) -> dict:
        """Возвращает финальный dict политики для колонки: {'class':..., extras}.

        Правила:
        - явное правило в policy побеждает всегда;
        - иначе берём предложенный класс, НО categorical_candidate → categorical_synth
          (реальные значения не сохраняем, синтезируем токены с сохранением
          кардинальности/долей);
        - categorical_keep недостижим без явного whitelist.
        """
        key = f"{schema}.{table}.{column}"
        if key in self._columns:
            entry = dict(self._columns[key])
            entry.setdefault("generator", proposed_gen)
            entry["source"] = "policy"
            return entry
        # Дефолт (whitelist): ничего не «кипаем».
        if proposed == "categorical_candidate":
            klass = "categorical_synth"
        elif proposed in VALID_CLASSES:
            klass = proposed
        else:
            klass = "sensitive"
        return {"class": klass, "generator": proposed_gen, "source": "auto"}

    def order_groups(self, schema: str, table: str) -> list[list[str]]:
        return list((self._tables.get(f"{schema}.{table}") or {}).get("order_groups") or [])

    def dependencies(self, schema: str, table: str) -> list[dict]:
        """Функциональные зависимости таблицы: [{determinant, dependent, order_by?, keep_representative?}]."""
        deps = (self._tables.get(f"{schema}.{table}") or {}).get("dependencies") or []
        out = []
        for d in deps:
            if not d.get("determinant") or not d.get("dependent"):
                continue
            out.append({
                "determinant": d["determinant"],
                "dependent": d["dependent"],
                "order_by": d.get("order_by"),
                "keep_representative": bool(d.get("keep_representative", True)),
            })
        return out


In [ ]:
%%writefile generated/agc_profiler/linter.py
"""Профиль-линтер — главная точка аудита утечек. Делаем строгим.

Правило: если колонка НЕ класса categorical_keep, но в её профиле оказались
реальные значения (values / most_common_vals / histogram / histogram_bounds /
реальные min/max) — это ошибка, падаем с явным сообщением ДО записи profile.json.
"""
from __future__ import annotations

# Ключи, несущие реальные значения строк. Разрешены ТОЛЬКО в categorical_keep.
# value_map — представители функциональных зависимостей (реальные значения dependent);
# он тоже допустим лишь в categorical_keep (keep_representative — это whitelist-opt-in).
FORBIDDEN_REAL_VALUE_KEYS = (
    "values", "value_map", "most_common_vals", "mcv",
    "histogram", "histogram_bounds",
    "min_real", "max_real", "min", "max",
)


class ProfileLeakError(Exception):
    """Линтер нашёл реальные значения в неразрешённой колонке."""


def check_profile(profile: dict) -> None:
    """Бросает ProfileLeakError со списком нарушений. None — если чисто."""
    violations: list[str] = []
    for table in profile.get("tables", []):
        schema = table.get("schema")
        tname = table.get("table")
        for col_name, col in (table.get("columns") or {}).items():
            klass = col.get("policy")
            if klass == "categorical_keep":
                continue
            leaked = [k for k in FORBIDDEN_REAL_VALUE_KEYS
                      if k in col and col[k] not in (None, [], {}, "")]
            for k in leaked:
                violations.append(
                    f"{schema}.{tname}.{col_name}: класс '{klass}' содержит реальные "
                    f"значения в поле '{k}' (разрешено только для categorical_keep)"
                )
    if violations:
        raise ProfileLeakError(
            "Линтер профиля: обнаружены потенциальные утечки реальных значений:\n  - "
            + "\n  - ".join(violations)
        )


In [ ]:
%%writefile generated/agc_profiler/profile_builder.py
"""Сборка profile.json из каталога (типы/хранение) + анализа СЭМПЛА в pandas.

Ключевые решения (по договорённости):
- PK не объявлен в DDL → выводим гипотезу по сэмплу (analyze.find_pk). FK не выводим.
- Вся статистика/категории/зависимости считаются в pandas на случайном сэмпле
  (до ~1M строк), а не отдельными GROUP BY к БД. Редкие категории могут потеряться.
- Функциональные зависимости (task_subtype → task_questionary): для каждой категории
  берём одного представителя dependent (свежайшая строка, где dependent не NULL).

Для каждой колонки собираем ТОЛЬКО поля, разрешённые её классом (whitelist):
- categorical_keep : реальные категории и доли (values), либо представители FD (value_map);
- categorical_synth: n_distinct + доли (freqs), БЕЗ реальных значений;
- sensitive_numeric: тип, precision/scale, null_frac, форма распределения, порядок среднего;
- pii             : тип, null_frac, генератор, unique_like — синтетика на выходе;
- key             : роль pk (гипотеза), n_distinct, null_frac;
- datetime        : тип, null_frac, синтетический диапазон, order_group.
"""
from __future__ import annotations

import math
from datetime import datetime, timezone

import pandas as pd
from sqlalchemy.engine import Engine

from agc_common import get_logger
from agc_profiler import analyze
from agc_profiler import catalog_reader as cat
from agc_profiler import classifier as clf
from agc_profiler.policy import Policy
from agc_profiler.sampler import sample_dataframe

log = get_logger("profiler.build")

PROFILE_VERSION = 2


def _coarse_magnitude(value: float) -> str | None:
    """Огрубление до порядка величины: 48213.7 -> '~5e4'. Не реальное значение строки."""
    try:
        v = abs(float(value))
    except (TypeError, ValueError):
        return None
    if v == 0:
        return "~0"
    exp = int(math.floor(math.log10(v)))
    lead = round(v / (10 ** exp))
    if lead == 10:
        lead, exp = 1, exp + 1
    return f"~{lead}e{exp}"


def _numeric_avg_hint(policy_entry: dict, series: "pd.Series | None") -> str | None:
    """avg_hint: из policy; иначе огрублённое среднее сэмпла (наружу — только порядок)."""
    if policy_entry.get("avg_hint"):
        return str(policy_entry["avg_hint"])
    if series is None:
        return None
    nums = pd.to_numeric(series, errors="coerce").dropna()
    if nums.empty:
        return None
    return _coarse_magnitude(float(nums.mean()))


def _rel_n_distinct(stats: dict, klass: str) -> float | int | None:
    """n_distinct для профиля. Категории: абсолютное число из сэмпла (не масштабируем).
    Прочее: относительная оценка (доля уникальных в сэмпле, со знаком минус)."""
    n_abs = stats.get("n_distinct")
    if klass in ("categorical_keep", "categorical_synth", "key"):
        return n_abs
    up = stats.get("unique_perc") or 0.0
    if up >= 99.0:
        return -1.0
    return -round(up / 100.0, 4) if up else n_abs


def _build_column(name: str, col_meta: dict, stats: dict, policy_entry: dict,
                  series: "pd.Series | None", is_pk: bool, order_group_cols: set,
                  dep_info: dict | None) -> dict:
    klass = policy_entry["class"]
    base = {"pg_type": col_meta["pg_type"], "policy": klass,
            "proposed_source": policy_entry.get("source"),
            "null_frac": round(stats.get("null_frac") or 0.0, 6),
            "not_null_perc": stats.get("not_null_perc"),
            "unique_perc": stats.get("unique_perc")}

    # Зависимая колонка (dependent) с сохранением представителя — особый случай.
    if dep_info and dep_info.get("value_map") is not None:
        base["policy"] = "categorical_keep"       # хранит реальные представители → whitelist
        base["depends_on"] = [dep_info["determinant"]]
        base["value_map"] = dep_info["value_map"]  # {категория: представитель}
        base["n_distinct"] = len(dep_info["value_map"])
        return base
    if dep_info and dep_info.get("value_map") is None:
        # Синтетический представитель на категорию (реальные значения не храним).
        base["policy"] = "categorical_synth"
        base["depends_on"] = [dep_info["determinant"]]
        base["dependent_synth"] = True
        base["n_distinct"] = dep_info.get("card")
        return base

    if klass == "categorical_keep":
        base["n_distinct"] = stats.get("n_distinct")
        base["values"] = stats.get("categories") or []
    elif klass == "categorical_synth":
        base["n_distinct"] = stats.get("n_distinct")
        freqs = [f for _, f in (stats.get("categories") or [])]
        if freqs:
            base["freqs"] = freqs
    elif klass == "sensitive_numeric":
        base["n_distinct"] = _rel_n_distinct(stats, klass)
        base["precision"] = col_meta.get("precision")
        base["scale"] = col_meta.get("scale")
        base["dist"] = policy_entry.get("dist", "lognormal")
        hint = _numeric_avg_hint(policy_entry, series)
        if hint:
            base["avg_hint"] = hint
    elif klass == "pii":
        up = stats.get("unique_perc") or 0.0
        base["generator"] = policy_entry.get("generator") or "text"
        base["unique_like"] = is_pk or up >= 90.0
        if col_meta.get("char_len"):
            base["char_len"] = col_meta["char_len"]
    elif klass == "key":
        base["role"] = "pk" if is_pk else "key"
        base["pk_inferred"] = bool(is_pk)
        base["n_distinct"] = _rel_n_distinct(stats, klass)
    elif klass == "datetime":
        base["range"] = policy_entry.get("range", {"start": "2018-01-01", "end": "2024-12-31"})
        if name in order_group_cols:
            base["order_group"] = True
    else:  # sensitive (generic)
        base["generator"] = policy_entry.get("generator") or "text"
    return base


def build_table_profile(engine: Engine, schema: str, table: str, policy: Policy,
                        *, sample_n: int = 1_000_000, timeout_ms: int | None = None) -> dict:
    meta = cat.read_table_meta(engine, schema, table)
    columns = cat.read_columns(engine, schema, table)
    dist_key = cat.read_distribution(engine, schema, table)
    part_keys = cat.read_partition_keys(engine, schema, table)

    df = sample_dataframe(engine, schema, table, sample_n,
                          est=meta.get("reltuples") or None, timeout_ms=timeout_ms)

    pk = analyze.find_pk(df) if not df.empty else []
    pk_set = set(pk)

    # Статистика + категории по каждой колонке из сэмпла.
    stats_by_col: dict[str, dict] = {}
    for col in columns:
        name = col["name"]
        s = analyze.column_stats(df, name)
        if 0 < s["n_distinct"] <= analyze.CATEGORICAL_MAX_DISTINCT:
            s["categories"] = analyze.top_categories(df, name)
        stats_by_col[name] = s

    # Функциональные зависимости (policy): представитель dependent на категорию.
    dep_by_col: dict[str, dict] = {}
    for dep in policy.dependencies(schema, table):
        det, dcol = dep["determinant"], dep["dependent"]
        rep = analyze.dependency_representatives(df, det, dcol, dep.get("order_by"))
        if not rep:
            log.warning("Зависимость %s→%s: представителей не найдено (пусто/нет колонок)", det, dcol)
            continue
        if dep["keep_representative"]:
            dep_by_col[dcol] = {"determinant": det, "value_map": rep}
        else:
            dep_by_col[dcol] = {"determinant": det, "value_map": None, "card": len(rep)}
        log.info("Зависимость %s→%s: %d категорий, keep_representative=%s",
                 det, dcol, len(rep), dep["keep_representative"])

    order_group_cols = {c for grp in policy.order_groups(schema, table) for c in grp}

    out_columns: dict[str, dict] = {}
    for col in columns:
        name = col["name"]
        stats = stats_by_col[name]
        sample_vals = (df[name].dropna().astype(str).head(30).tolist()
                       if (not df.empty and name in df.columns) else [])
        proposed, gen = clf.propose(name, col["pg_type"], is_pk=name in pk_set,
                                    n_distinct=stats.get("n_distinct"), sample_values=sample_vals)
        policy_entry = policy.resolve(schema, table, name, proposed, gen)
        series = df[name] if (not df.empty and name in df.columns) else None
        out_columns[name] = _build_column(name, col, stats, policy_entry, series,
                                           name in pk_set, order_group_cols, dep_by_col.get(name))

    profile = {
        "schema": schema, "table": table,
        "relkind": meta["relkind"], "is_view": meta["is_view"], "storage": meta["storage"],
        "distributed_by": dist_key, "partitioned_by": part_keys,
        "row_count": {"value": meta["reltuples"], "estimated": meta["row_count_estimated"],
                      "sample_rows": int(len(df))},
        "pk_hypothesis": pk,
        "column_dependencies": [{"determinant": d["determinant"], "dependent": dcol}
                                for dcol, d in dep_by_col.items()],
        "defaults": {c["name"]: c["default"] for c in columns if c["default"]},
        "not_null": [c["name"] for c in columns if not c["nullable"]],
        "columns": out_columns,
    }
    log.info("Профиль %s.%s: %d колонок, pk=%s, storage=%s, sample=%d строк",
             schema, table, len(out_columns), pk or "-", meta["storage"], len(df))
    return profile


def build_profile(engine: Engine, tables: list[tuple[str, str]], policy: Policy,
                  *, sample_n: int = 1_000_000, timeout_ms: int | None = None) -> dict:
    table_profiles = [build_table_profile(engine, s, t, policy,
                                          sample_n=sample_n, timeout_ms=timeout_ms)
                      for s, t in tables]
    return {
        "profile_version": PROFILE_VERSION,
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_dialect": "greenplum",
        "note": "Синтетический профиль: реальные значения только в categorical_keep. "
                "PK — гипотеза по сэмплу; FK не выводится.",
        "tables": table_profiles,
    }


In [ ]:
%%writefile generated/agc_profiler/cli.py
"""CLI программы 1 (profiler). Закрытый контур.

Пример:
    python -m agc_profiler.cli --tables "public.tasks,public.clients" \
        --policy policy.yaml --out profile.json --sample

    python -m agc_profiler.cli --tables-csv tables.csv --out profile.json
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

from agc_common import get_logger
from agc_profiler.db import make_engine
from agc_profiler.linter import check_profile
from agc_profiler.policy import Policy
from agc_profiler.profile_builder import build_profile

log = get_logger("profiler.cli")


def parse_tables_arg(value: str) -> list[tuple[str, str]]:
    """'schema.table,schema.table2' -> [(schema, table), ...]."""
    out = []
    for item in value.split(","):
        item = item.strip()
        if not item:
            continue
        if "." not in item:
            raise ValueError(f"Ожидался формат schema.table, получено: {item!r}")
        schema, table = item.split(".", 1)
        out.append((schema.strip(), table.strip()))
    return out


def parse_tables_csv(path: str | Path) -> list[tuple[str, str]]:
    """CSV со списком таблиц. Дефолт-колонки: schema,table. Поддерживаем также
    schema_name,table_name (как tables_list.csv проекта)."""
    rows: list[tuple[str, str]] = []
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        cols = {c.lower(): c for c in (reader.fieldnames or [])}
        sc = cols.get("schema") or cols.get("schema_name")
        tc = cols.get("table") or cols.get("table_name")
        if not sc or not tc:
            raise ValueError("CSV должен содержать колонки schema,table (или schema_name,table_name)")
        for r in reader:
            schema, table = (r[sc] or "").strip(), (r[tc] or "").strip()
            if schema and table:
                rows.append((schema, table))
    return rows


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="agc_profiler", description="Profiler (закрытый контур)")
    src = p.add_mutually_exclusive_group(required=True)
    src.add_argument("--tables", help="schema.table через запятую")
    src.add_argument("--tables-csv", help="CSV со списком таблиц (колонки schema,table)")
    p.add_argument("--policy", help="YAML policy-файл (whitelist чувствительности)")
    p.add_argument("--out", default="profile.json", help="куда писать профиль")
    p.add_argument("--dsn", help="DSN БД (иначе AGC_DB_DSN/DB_DSN/db_config.json)")
    p.add_argument("--db-config", help="db_config.json с параметрами подключения")
    p.add_argument("--sample-n", type=int, default=1_000_000,
                   help="сколько строк тянуть в сэмпл на таблицу (pandas в памяти)")
    p.add_argument("--statement-timeout-ms", type=int, default=600_000)
    return p


def main(argv: list[str] | None = None) -> int:
    args = build_arg_parser().parse_args(argv)
    tables = parse_tables_csv(args.tables_csv) if args.tables_csv else parse_tables_arg(args.tables)
    if not tables:
        log.error("Список таблиц пуст."); return 2
    log.info("К обработке %d таблиц: %s", len(tables),
             ", ".join(f"{s}.{t}" for s, t in tables[:10]))

    engine = make_engine(args.dsn, args.db_config, args.statement_timeout_ms)
    policy = Policy.load(args.policy)
    profile = build_profile(engine, tables, policy, sample_n=args.sample_n,
                            timeout_ms=args.statement_timeout_ms)

    # Линтер — строгая проверка перед записью (главная точка аудита утечек).
    check_profile(profile)
    log.info("Линтер пройден: реальные значения только в categorical_keep.")

    Path(args.out).write_text(
        json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("Профиль записан: %s (%d таблиц)", args.out, len(profile["tables"]))
    return 0


if __name__ == "__main__":
    sys.exit(main())


### Программа 2 — generator (открытый контур)

Модули: `profile_parser` · `ddl_builder` (schema.sql, топосорт по FK) · `value_generators` (значения по классам) · `key_linker` (PK/FK, масштабирование) · `writer` (CSV/INSERT) · `cli`.

In [ ]:
%%writefile generated/agc_generator/__init__.py
"""Программа 2 — generator (открытый контур).

Читает profile.json, строит schema.sql (DDL) и синтетические данные (CSV или
INSERT-ы). Лёгкий детерминированный генератор по профилю — без ML-синтезаторов
(SDV/CTGAN и т.п.), чтобы физически не воспроизводить реальные совместные
распределения и не «запоминать» реальные строки.
"""


In [ ]:
%%writefile generated/agc_generator/profile_parser.py
"""Парсер profile.json в удобные структуры для генератора.

FK в профиле нет (по договорённости — джойны подбираются позже). Есть PK-гипотеза
и функциональные зависимости (column_dependencies).
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class Column:
    name: str
    pg_type: str
    policy: str
    raw: dict = field(default_factory=dict)

    def get(self, key, default=None):
        return self.raw.get(key, default)


@dataclass
class Table:
    schema: str
    table: str
    is_view: bool
    storage: str
    distributed_by: list
    partitioned_by: list
    row_count: int
    row_count_estimated: bool
    pk_hypothesis: list
    column_dependencies: list  # [{determinant, dependent}]
    defaults: dict
    not_null: list
    columns: dict  # name -> Column

    @property
    def fqn(self) -> str:
        return f"{self.schema}.{self.table}"

    @property
    def pk(self) -> list:
        return self.pk_hypothesis or []


@dataclass
class Profile:
    version: int
    tables: list  # list[Table]

    def by_fqn(self) -> dict:
        return {t.fqn: t for t in self.tables}


def load_profile(path: str | Path) -> Profile:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    tables = []
    for t in data.get("tables", []):
        columns = {
            name: Column(name=name, pg_type=c.get("pg_type", "text"),
                         policy=c.get("policy", "sensitive"), raw=c)
            for name, c in (t.get("columns") or {}).items()
        }
        rc = t.get("row_count") or {}
        tables.append(Table(
            schema=t["schema"], table=t["table"],
            is_view=bool(t.get("is_view")), storage=t.get("storage", "heap"),
            distributed_by=t.get("distributed_by") or [],
            partitioned_by=t.get("partitioned_by") or [],
            row_count=int(rc.get("value") or 0),
            row_count_estimated=bool(rc.get("estimated", True)),
            pk_hypothesis=t.get("pk_hypothesis") or [],
            column_dependencies=t.get("column_dependencies") or [],
            defaults=t.get("defaults") or {},
            not_null=t.get("not_null") or [],
            columns=columns,
        ))
    return Profile(version=int(data.get("profile_version", 2)), tables=tables)


In [ ]:
%%writefile generated/agc_generator/ddl_builder.py
"""DDL-билдер: CREATE TABLE из профиля (не завися от pg_dump).

Воспроизводим типы, NOT NULL, DEFAULT и PK (гипотеза по сэмплу — удобно агенту,
хотя на реальных данных ключ не объявлен). FK НЕ создаём: связей в профиле нет,
ключи джойнов подбираются позже на синтетике. По умолчанию тестовая БД — heap,
одиночный сегмент; флаг keep_gpdb сохраняет DISTRIBUTED BY и заметку о партициях.
Вью материализуем как обычные таблицы-заглушки.
"""
from __future__ import annotations

from agc_common import get_logger, quote_ident
from agc_generator.profile_parser import Profile, Table

log = get_logger("generator.ddl")


def _column_ddl(table: Table, name: str) -> str:
    col = table.columns[name]
    parts = [f"  {quote_ident(name)} {col.pg_type}"]
    if name in table.not_null and name not in table.pk:
        parts.append("NOT NULL")
    default = table.defaults.get(name)
    # nextval(...) отбрасываем: секвенций в тестовой БД нет, PK генерируем сами.
    if default and "nextval(" not in str(default).lower():
        parts.append(f"DEFAULT {default}")
    return " ".join(parts)


def build_table_ddl(table: Table, keep_gpdb: bool) -> str:
    lines = [_column_ddl(table, name) for name in table.columns]
    if table.pk:
        cols = ", ".join(quote_ident(c) for c in table.pk)
        lines.append(f"  PRIMARY KEY ({cols})  -- гипотеза по сэмплу")

    ddl = (
        f"CREATE TABLE {quote_ident(table.schema)}.{quote_ident(table.table)} (\n"
        + ",\n".join(lines) + "\n)"
    )
    if keep_gpdb and table.distributed_by:
        cols = ", ".join(quote_ident(c) for c in table.distributed_by)
        ddl += f"\nDISTRIBUTED BY ({cols})"
    elif keep_gpdb:
        ddl += "\nDISTRIBUTED RANDOMLY"
    ddl += ";"
    if keep_gpdb and table.partitioned_by:
        ddl += (f"\n-- NOTE: исходная таблица партиционирована по "
                f"{', '.join(table.partitioned_by)}; для теста партиции упрощены.")
    return ddl


def build_ddl(profile: Profile, *, keep_gpdb: bool = False) -> str:
    tables = list(profile.tables)
    views = [t for t in tables if t.is_view]
    if views:
        log.info("Вью в профиле (%d) материализуем как таблицы-заглушки: %s",
                 len(views), ", ".join(v.fqn for v in views))
    schemas = sorted({t.schema for t in tables})
    header = [
        "-- Автосгенерированный DDL тестовой БД (air-gap clone).",
        "-- Реальные данные не воспроизводятся; структура — из profile.json.",
        "-- PK — гипотеза по сэмплу; FK не создаются (подбираются на синтетике).",
        "",
    ]
    for s in schemas:
        header.append(f"CREATE SCHEMA IF NOT EXISTS {quote_ident(s)};")
    header.append("")
    body = [build_table_ddl(t, keep_gpdb) for t in tables]
    return "\n".join(header) + "\n\n".join(body) + "\n"


In [ ]:
%%writefile generated/agc_generator/value_generators.py
"""Генераторы значений по классу колонки.

Лёгкие, детерминированные (по seed), без ML-синтезаторов. numpy/Faker
опциональны — при их отсутствии работает stdlib-реализация (random/math).

Классы: categorical_keep, categorical_synth, sensitive_numeric, pii, datetime,
sensitive (generic). Ключи (key) обрабатывает key_linker.
"""
from __future__ import annotations

import math
import random
from datetime import date, datetime, timedelta

from agc_common import get_logger, stable_hash

log = get_logger("generator.values")

try:  # опционально — красивее pii; при отсутствии работает встроенный фолбэк
    from faker import Faker  # type: ignore
    _FAKER = Faker("ru_RU")
except Exception:  # noqa: BLE001
    _FAKER = None

# --- встроенные словари для pii-фолбэка (без внешних зависимостей) ---
# Значения ЯВНО синтетические (в духе «Иванов Иван Иванович», «ИНН 1234567890»),
# чтобы сразу было видно: это не реальные данные.
_FIRST_M = ["Александр", "Дмитрий", "Максим", "Сергей", "Андрей", "Алексей", "Иван", "Кирилл"]
_FIRST_F = ["Анна", "Елена", "Ольга", "Наталья", "Мария", "Татьяна", "Ирина", "Екатерина"]
_PATR_M = ["Иванович", "Петрович", "Сергеевич", "Андреевич", "Алексеевич", "Дмитриевич"]
_PATR_F = ["Ивановна", "Петровна", "Сергеевна", "Андреевна", "Алексеевна", "Дмитриевна"]
_LAST = ["Иванов", "Петров", "Смирнов", "Кузнецов", "Соколов", "Попов", "Лебедев", "Козлов"]
_STREETS = ["Ленина", "Мира", "Советская", "Гагарина", "Победы", "Садовая", "Лесная"]
_COMPANY = ["Ромашка", "Импульс", "Вектор", "Горизонт", "Альфа", "Дельта", "Пример", "Ресурс"]
_COMPANY_FORM = ["ООО", "АО", "ПАО", "ЗАО"]


def _apply_nulls(values: list, null_frac: float, rng: random.Random) -> list:
    if null_frac <= 0:
        return values
    return [None if rng.random() < null_frac else v for v in values]


# --------------------------------------------------------------------------- #
# Категориальные                                                              #
# --------------------------------------------------------------------------- #
def _weighted_all_present(labels: list, weights: list, n: int, rng: random.Random) -> list:
    """Сэмпл по весам, но КАЖДАЯ категория присутствует хотя бы раз (если n>=|labels|).
    Так все категории попадают в синтетику — важно для GROUP BY по редким значениям."""
    labels = labels or ["__empty__"]
    weights = [max(0.0, float(w)) for w in weights] or [1.0]
    total = sum(weights) or 1.0
    weights = [w / total for w in weights]
    if n <= 0:
        return []
    if n >= len(labels):
        out = list(labels)  # по одному каждой категории
        out += rng.choices(labels, weights=weights, k=n - len(labels))
        rng.shuffle(out)
    else:
        out = rng.choices(labels, weights=weights, k=n)
    return out


def gen_categorical_keep(values: list, n: int, null_frac: float, rng: random.Random) -> list:
    """Сэмпл из реальных distinct-значений по их весам — сохраняет кардинальность и
    распределение групп; каждая категория присутствует хотя бы раз."""
    labels = [v[0] for v in values]
    weights = [v[1] for v in values]
    out = _weighted_all_present(labels, weights, n, rng)
    return _apply_nulls(out, null_frac, rng)


def gen_categorical_synth(k: int, freqs: list | None, n: int, null_frac: float,
                          rng: random.Random, prefix: str = "cat") -> list:
    """K синтетических токенов, сэмпл по весам freqs (или равномерно); каждая
    категория присутствует хотя бы раз. Реальные метки не используются."""
    k = max(1, k)
    tokens = [f"{prefix}_{i:04d}" for i in range(k)]
    if freqs:
        w = [max(0.0, float(x)) for x in freqs[:k]]
        while len(w) < k:
            w.append(min(w) if w else 1.0)
    else:
        w = [1.0] * k
    out = _weighted_all_present(tokens, w, n, rng)
    return _apply_nulls(out, null_frac, rng)


def gen_dependent_keep(determinant_values: list, value_map: dict, rng: random.Random,
                       null_frac: float = 0.0) -> list:
    """Зависимая колонка с реальными представителями: dependent = value_map[determinant].
    «Опросник» остаётся привязан к своему подтипу. Неизвестная категория → None."""
    out = [value_map.get(str(d)) if d is not None else None for d in determinant_values]
    return _apply_nulls(out, null_frac, rng)


def gen_dependent_synth(determinant_values: list, rng: random.Random,
                        prefix: str = "dep", null_frac: float = 0.0) -> list:
    """Зависимая колонка с СИНТЕТИЧЕСКИМ представителем: стабильный токен на категорию
    (одинаковый для одной категории), реальные значения не используются."""
    out = [None if d is None else f"{prefix}_{stable_hash(prefix, d) % 100000:05d}"
           for d in determinant_values]
    return _apply_nulls(out, null_frac, rng)


# --------------------------------------------------------------------------- #
# Числа                                                                       #
# --------------------------------------------------------------------------- #
def _parse_magnitude(avg_hint: str | None, default: float = 1000.0) -> float:
    if not avg_hint:
        return default
    s = str(avg_hint).lstrip("~").strip()
    try:
        return float(s)
    except ValueError:
        return default


def gen_sensitive_numeric(spec: dict, n: int, rng: random.Random) -> list:
    """Числа из указанной формы распределения. Правдоподобно для агрегатов,
    но не похоже на реальные суммы (форма и порядок — из профиля, значения — новые)."""
    dist = (spec.get("dist") or "lognormal").lower()
    scale = spec.get("scale")
    mean = _parse_magnitude(spec.get("avg_hint"), default=1000.0)
    null_frac = float(spec.get("null_frac") or 0.0)

    out: list = []
    if dist == "lognormal":
        sigma = 1.0
        mu = math.log(mean) - 0.5 * sigma * sigma if mean > 0 else 0.0
        for _ in range(n):
            out.append(math.exp(rng.gauss(mu, sigma)))
    elif dist == "normal":
        sigma = max(1.0, mean * 0.3)
        for _ in range(n):
            out.append(rng.gauss(mean, sigma))
    else:  # uniform
        lo, hi = 0.0, max(1.0, mean * 2)
        for _ in range(n):
            out.append(rng.uniform(lo, hi))

    # precision/scale и целочисленность.
    base = (spec.get("pg_type") or "").split("(")[0].strip().lower()
    is_int = base in ("smallint", "integer", "bigint")
    rounded = []
    for v in out:
        if is_int:
            rounded.append(int(round(v)))
        elif scale is not None:
            rounded.append(round(v, int(scale)))
        else:
            rounded.append(round(v, 2))
    return _apply_nulls(rounded, null_frac, rng)


# --------------------------------------------------------------------------- #
# PII                                                                         #
# --------------------------------------------------------------------------- #
def _pii_one(generator: str, rng: random.Random, seq: int) -> str:
    if _FAKER is not None:
        try:
            return _faker_value(generator, seq)
        except Exception:  # noqa: BLE001
            pass
    return _builtin_pii(generator, rng, seq)


def _faker_value(generator: str, seq: int) -> str:
    f = _FAKER
    mapping = {
        "full_name": f.name, "email": f.email, "phone": f.phone_number,
        "address": f.address, "company": f.company,
    }
    if generator in mapping:
        return str(mapping[generator]()).replace("\n", ", ")
    if generator == "inn":
        return f.numerify("##########")
    if generator == "snils":
        return f.numerify("###-###-### ##")
    if generator == "account":
        return f.numerify("#" * 20)
    if generator == "passport":
        return f.numerify("#### ######")
    return f"{generator}_{seq:06d}"


def _builtin_pii(generator: str, rng: random.Random, seq: int) -> str:
    if generator == "full_name":
        # ФИО с отчеством: «Иванов Иван Иванович» / «Иванова Анна Ивановна».
        if rng.random() < 0.5:
            return f"{rng.choice(_LAST)} {rng.choice(_FIRST_M)} {rng.choice(_PATR_M)}"
        return f"{rng.choice(_LAST)}а {rng.choice(_FIRST_F)} {rng.choice(_PATR_F)}"
    if generator == "email":
        return f"user{seq:06d}@example.test"
    if generator == "phone":
        return "+7" + "".join(str(rng.randint(0, 9)) for _ in range(10))
    if generator == "inn":
        return "".join(str(rng.randint(0, 9)) for _ in range(10))
    if generator == "snils":
        d = "".join(str(rng.randint(0, 9)) for _ in range(11))
        return f"{d[0:3]}-{d[3:6]}-{d[6:9]} {d[9:11]}"
    if generator == "account":
        return "".join(str(rng.randint(0, 9)) for _ in range(20))
    if generator == "passport":
        return "".join(str(rng.randint(0, 9)) for _ in range(4)) + " " + \
               "".join(str(rng.randint(0, 9)) for _ in range(6))
    if generator == "address":
        return f"ул. {rng.choice(_STREETS)}, д. {rng.randint(1, 120)}, кв. {rng.randint(1, 200)}"
    if generator == "company":
        return f'{rng.choice(_COMPANY_FORM)} "{rng.choice(_COMPANY)}"'
    # generic text
    return f"{generator}_{seq:06d}"


def gen_pii(generator: str, n: int, null_frac: float, unique_like: bool,
            rng: random.Random) -> list:
    out: list = []
    seen: set = set()
    seq = 0
    while len(out) < n:
        seq += 1
        val = _pii_one(generator, rng, seq)
        if unique_like:
            if val in seen:
                val = f"{val}#{seq}"  # гарантируем уникальность
            seen.add(val)
        out.append(val)
    return _apply_nulls(out, null_frac, rng)


# --------------------------------------------------------------------------- #
# Даты                                                                        #
# --------------------------------------------------------------------------- #
def _parse_date(s: str, default: date) -> date:
    try:
        return datetime.strptime(str(s)[:10], "%Y-%m-%d").date()
    except (ValueError, TypeError):
        return default


def gen_datetime(spec: dict, n: int, rng: random.Random) -> list:
    rng_spec = spec.get("range") or {}
    start = _parse_date(rng_spec.get("start"), date(2018, 1, 1))
    end = _parse_date(rng_spec.get("end"), date(2024, 12, 31))
    span = max(1, (end - start).days)
    is_ts = "timestamp" in (spec.get("pg_type") or "").lower()
    out: list = []
    for _ in range(n):
        d = start + timedelta(days=rng.randint(0, span))
        if is_ts:
            out.append(datetime(d.year, d.month, d.day,
                                rng.randint(0, 23), rng.randint(0, 59), rng.randint(0, 59)))
        else:
            out.append(d)
    return _apply_nulls(out, float(spec.get("null_frac") or 0.0), rng)


def gen_generic(spec: dict, n: int, rng: random.Random) -> list:
    """sensitive (generic) — короткие псевдо-текстовые токены."""
    gen = spec.get("generator") or "val"
    out = [f"{gen}_{i:06d}" for i in range(n)]
    rng.shuffle(out)
    return _apply_nulls(out, float(spec.get("null_frac") or 0.0), rng)


In [ ]:
%%writefile generated/agc_generator/key_linker.py
"""Генерация строк таблицы: суррогатный PK, категории со всеми значениями,
функциональные зависимости. FK НЕ используется — таблицы независимы (ключи
джойнов подбираются позже на синтетике).

Масштабирование числа строк: N = max(1, round(row_count * scale)). Кардинальность
категорий сохраняем (все категории домена), не масштабируем вниз.
"""
from __future__ import annotations

import random

from agc_common import get_logger, stable_hash
from agc_generator import value_generators as vg
from agc_generator.profile_parser import Profile, Table

log = get_logger("generator.keys")


def scaled_rowcount(row_count: int, scale: float) -> int:
    return max(1, int(round(row_count * scale)))


# Абсолютные n_distinct до этого — «категориальная» кардинальность домена
# (коды/статусы/флаги), её НЕ масштабируем вниз (иначе GROUP BY схлопнется).
CATEGORICAL_ABS_MAX = 200


def categorical_cardinality(n_distinct, freqs, n_rows: int) -> int:
    """Число distinct для categorical_synth: сохраняем кардинальность домена
    (зажато в [1, n_rows]); число корзин freqs — естественный пол."""
    floor = len(freqs) if freqs else 0
    if n_distinct is None:
        k = floor or n_rows
    elif n_distinct < 0:
        k = round(-float(n_distinct) * n_rows)
    else:
        k = int(round(n_distinct))
    return max(1, min(max(k, floor), n_rows))


def _col_seed(base_seed: int, table: Table, col: str) -> int:
    return (stable_hash(table.fqn, col) ^ base_seed) & 0x7FFFFFFF


def _gen_pk_pool(table: Table, pk_col: str, n_rows: int) -> list:
    """Суррогатный PK: 1..N для числовых, иначе синтетические уникальные токены."""
    col = table.columns.get(pk_col)
    base = (col.pg_type.split("(")[0].strip().lower() if col else "bigint")
    if base in ("smallint", "integer", "bigint", "numeric", "decimal"):
        return list(range(1, n_rows + 1))
    return [f"{table.table[:8]}_{i:06d}" for i in range(1, n_rows + 1)]


def _column_order(table: Table, dep_map: dict) -> list[str]:
    """Порядок генерации: детерминант раньше зависимой колонки."""
    names = list(table.columns.keys())
    placed, order = set(), []

    def place(name):
        if name in placed:
            return
        det = dep_map.get(name)
        if det and det in table.columns and det not in placed:
            place(det)
        placed.add(name)
        order.append(name)

    for n in names:
        place(n)
    return order


def generate(profile: Profile, scale: float, seed: int) -> dict[str, list[dict]]:
    """{fqn: [row_dict, ...]}. Таблицы независимы; зависимости внутри таблицы соблюдены."""
    data: dict[str, list[dict]] = {}
    for table in profile.tables:
        n_rows = scaled_rowcount(table.row_count, scale)
        dep_map = {d["dependent"]: d["determinant"] for d in table.column_dependencies}
        order = _column_order(table, dep_map)
        cols_out: dict[str, list] = {}

        for name in order:
            col = table.columns[name]
            spec = dict(col.raw)
            spec.setdefault("pg_type", col.pg_type)
            klass = col.policy
            crng = random.Random(_col_seed(seed, table, name))
            null_frac = float(spec.get("null_frac") or 0.0)

            # Зависимая колонка (task_questionary): значение определяется детерминантом.
            if name in dep_map:
                det = dep_map[name]
                det_vals = cols_out.get(det) or [None] * n_rows
                vmap = spec.get("value_map")
                det_is_keep = table.columns.get(det) and table.columns[det].policy == "categorical_keep"
                if vmap is not None and det_is_keep:
                    cols_out[name] = vg.gen_dependent_keep(
                        det_vals, {str(k): v for k, v in vmap.items()}, crng, null_frac)
                else:
                    cols_out[name] = vg.gen_dependent_synth(
                        det_vals, crng, prefix=name[:8], null_frac=null_frac)
                continue

            if name in table.pk and klass == "key":
                cols_out[name] = _gen_pk_pool(table, name, n_rows)
            elif klass == "key":
                cols_out[name] = list(range(1, n_rows + 1))
            elif klass == "categorical_keep":
                cols_out[name] = vg.gen_categorical_keep(
                    spec.get("values") or [], n_rows, null_frac, crng)
            elif klass == "categorical_synth":
                k = categorical_cardinality(spec.get("n_distinct"), spec.get("freqs"), n_rows)
                cols_out[name] = vg.gen_categorical_synth(
                    k, spec.get("freqs"), n_rows, null_frac, crng, prefix=name[:8])
            elif klass == "sensitive_numeric":
                cols_out[name] = vg.gen_sensitive_numeric(spec, n_rows, crng)
            elif klass == "pii":
                cols_out[name] = vg.gen_pii(
                    spec.get("generator") or "text", n_rows, null_frac,
                    bool(spec.get("unique_like")), crng)
            elif klass == "datetime":
                cols_out[name] = vg.gen_datetime(spec, n_rows, crng)
            else:
                cols_out[name] = vg.gen_generic(spec, n_rows, crng)

        rows = _to_rows(table, cols_out, n_rows)
        rows = _enforce_date_order(table, rows)
        data[table.fqn] = rows
        log.info("Данные %s: %d строк, %d колонок (scale=%s)",
                 table.fqn, n_rows, len(cols_out), scale)
    return data


def _to_rows(table: Table, cols_out: dict[str, list], n_rows: int) -> list[dict]:
    names = list(table.columns.keys())
    return [{name: cols_out.get(name, [None] * n_rows)[i] for name in names}
            for i in range(n_rows)]


def _enforce_date_order(table: Table, rows: list[dict]) -> list[dict]:
    """Внутри order_group даты по возрастанию (created <= updated <= ...)."""
    groups = [name for name, col in table.columns.items() if col.get("order_group")]
    if len(groups) < 2:
        return rows
    for row in rows:
        vals = sorted(row[name] for name in groups if row[name] is not None)
        j = 0
        for name in groups:
            if row[name] is not None:
                row[name] = vals[j]
                j += 1
    return rows


In [ ]:
%%writefile generated/agc_generator/writer.py
"""Запись синтетических данных: CSV на таблицу или батч INSERT-ов."""
from __future__ import annotations

import csv
from datetime import date, datetime
from pathlib import Path

from agc_common import get_logger, quote_ident
from agc_generator.profile_parser import Profile

log = get_logger("generator.writer")


def _cell_csv(v) -> str:
    if v is None:
        return ""
    if isinstance(v, (datetime, date)):
        return v.isoformat(sep=" ") if isinstance(v, datetime) else v.isoformat()
    return str(v)


def write_csv(profile: Profile, data: dict[str, list[dict]], outdir: Path) -> list[Path]:
    outdir.mkdir(parents=True, exist_ok=True)
    written = []
    for table in profile.tables:
        rows = data.get(table.fqn, [])
        path = outdir / f"{table.schema}.{table.table}.csv"
        cols = list(table.columns.keys())
        with open(path, "w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            w.writerow(cols)
            for row in rows:
                w.writerow([_cell_csv(row.get(c)) for c in cols])
        written.append(path)
        log.info("CSV %s: %d строк", path.name, len(rows))
    return written


def _cell_sql(v) -> str:
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float)):
        return repr(v)
    if isinstance(v, datetime):
        return "'" + v.isoformat(sep=" ") + "'"
    if isinstance(v, date):
        return "'" + v.isoformat() + "'"
    return "'" + str(v).replace("'", "''") + "'"


def write_sql(profile: Profile, data: dict[str, list[dict]], out_path: Path,
              batch: int = 500) -> Path:
    lines = ["-- Синтетические данные (air-gap clone). FK нет — порядок вставки любой.", ""]
    for table in profile.tables:
        rows = data.get(table.fqn, [])
        if not rows:
            continue
        cols = list(table.columns.keys())
        collist = ", ".join(quote_ident(c) for c in cols)
        target = f"{quote_ident(table.schema)}.{quote_ident(table.table)}"
        for i in range(0, len(rows), batch):
            chunk = rows[i:i + batch]
            values = ",\n  ".join(
                "(" + ", ".join(_cell_sql(r.get(c)) for c in cols) + ")" for r in chunk)
            lines.append(f"INSERT INTO {target} ({collist}) VALUES\n  {values};")
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    log.info("SQL данные: %s", out_path.name)
    return out_path


In [ ]:
%%writefile generated/agc_generator/cli.py
"""CLI программы 2 (generator). Открытый контур.

Пример:
    python -m agc_generator.cli --profile profile.json --scale 0.001 \
        --seed 42 --format csv --out out/

    python -m agc_generator.cli --profile profile.json --format sql --keep-gpdb
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

from agc_common import get_logger
from agc_generator.ddl_builder import build_ddl
from agc_generator.key_linker import generate
from agc_generator.profile_parser import load_profile
from agc_generator.writer import write_csv, write_sql

log = get_logger("generator.cli")


def build_arg_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="agc_generator", description="Generator (открытый контур)")
    p.add_argument("--profile", required=True, help="profile.json от profiler")
    p.add_argument("--scale", type=float, default=0.001,
                   help="scale_factor: доля от исходного числа строк (1.2M * 0.001 ~ 1200)")
    p.add_argument("--seed", type=int, default=42, help="seed для детерминизма")
    p.add_argument("--format", choices=("csv", "sql"), default="csv",
                   help="csv (файл на таблицу) или sql (батч INSERT-ов)")
    p.add_argument("--out", default="out", help="каталог/файл вывода")
    p.add_argument("--keep-gpdb", action="store_true",
                   help="сохранить GPDB-специфику (DISTRIBUTED BY, заметки о партициях)")
    return p


def main(argv: list[str] | None = None) -> int:
    args = build_arg_parser().parse_args(argv)
    profile = load_profile(args.profile)
    log.info("Профиль загружен: %d таблиц. scale=%s seed=%s format=%s",
             len(profile.tables), args.scale, args.seed, args.format)

    outdir = Path(args.out)
    outdir.mkdir(parents=True, exist_ok=True)

    # 1) DDL.
    ddl = build_ddl(profile, keep_gpdb=args.keep_gpdb)
    ddl_path = outdir / "schema.sql"
    ddl_path.write_text(ddl, encoding="utf-8")
    log.info("DDL записан: %s", ddl_path)

    # 2) Данные.
    data = generate(profile, args.scale, args.seed)
    if args.format == "csv":
        write_csv(profile, data, outdir / "data")
    else:
        write_sql(profile, data, outdir / "data.sql")

    total = sum(len(v) for v in data.values())
    log.info("Готово: %d таблиц, %d строк суммарно в %s", len(data), total, outdir)
    return 0


if __name__ == "__main__":
    sys.exit(main())


### Точки входа и вспомогательные файлы

In [ ]:
%%writefile generated/profiler.py
#!/usr/bin/env python3
"""Точка входа программы 1 (закрытый контур). См. agc_profiler/cli.py.

Запуск:  python profiler.py --tables "public.tasks,public.clients" --out profile.json
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from agc_profiler.cli import main  # noqa: E402

if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile generated/generator.py
#!/usr/bin/env python3
"""Точка входа программы 2 (открытый контур). См. agc_generator/cli.py.

Запуск:  python generator.py --profile profile.json --scale 0.001 --format csv --out out/
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from agc_generator.cli import main  # noqa: E402

if __name__ == "__main__":
    sys.exit(main())


### Документация, policy, примеры

In [ ]:
%%writefile generated/requirements.txt
# Минимальные зависимости. На закрытом контуре интернета нет — ставьте из
# заранее скачанных wheels (pip install --no-index --find-links=./wheels -r requirements.txt).

# --- Программа 1 (profiler, закрытый контур) ---
SQLAlchemy>=2.0        # адаптер БД
psycopg2-binary        # драйвер Greenplum/Postgres (или psycopg2 из системного пакета)
PyYAML>=6.0            # policy-файл
pandas>=1.5            # вся статистика профиля считается на сэмпле в pandas

# --- Программа 2 (generator, открытый контур) ---
# Ничего обязательного сверх stdlib. Ниже — опциональные улучшения:
# Faker>=20.0          # красивее pii; без него работает встроенный фолбэк
# numpy>=1.24          # не требуется: числа генерируются через stdlib random/math


In [ ]:
%%writefile generated/README.md
# air-gap DB clone — profiler + generator

Инструмент «безопасного клонирования» БД через воздушный зазор из **двух отдельных
программ**. Через зазор едет **не выгрузка данных, а маленький человекочитаемый
`profile.json`** (схема + статистика) — его можно глазами проверить перед переносом.

```
закрытый контур (реальная БД)                 открытый контур (LLM, без реальной БД)
┌───────────────────────────┐   profile.json  ┌────────────────────────────────────┐
│  profiler                 │  ─────────────▶ │  generator                         │
│  каталог + сэмпл (pandas)  │   (+ policy)    │  schema.sql + синтетические данные  │
│  → profile.json (аудит)    │                 │  (CSV или INSERT-ы)                │
└───────────────────────────┘                 └────────────────────────────────────┘
```

СУБД: **Greenplum** (MPP-форк PostgreSQL). Код совместим с разными версиями и мягко
деградирует, если GPDB-специфики (`gp_distribution_policy`, `pg_partitions`,
`pg_appendonly`) нет. Модуль **самостоятельный** — ничего из внешних проектов не тянет.

---

## Что переносить между контурами

Наружу едет **только `profile.json`**. Реальные строки данных не выгружаются:
реальные значения попадают в профиль лишь для колонок, явно одобренных в policy как
`categorical_keep` (и представителей зависимостей с `keep_representative`). Линтер
строго проверяет это перед записью.

---

## Ключевые решения

- **PK не объявлен в DDL** → выводим **гипотезу по сэмплу** (минимальная уникальная
  комбинация, как в исходном проекте). **FK не выводим вообще** — ключи джойнов
  подбираются позже уже на синтетике.
- **Вся статистика считается в pandas на случайном сэмпле** (до ~1M строк на таблицу),
  а не отдельными `GROUP BY` к БД. Редкие категории в сэмпле могут потеряться — это
  допустимо.
- **Категория = меньше 200 уникальных значений** в сэмпле. Все попавшие в сэмпл
  категории и их доли сохраняются; генератор гарантирует, что **каждая категория
  присутствует** в синтетике (для корректных GROUP BY).
- **Функциональные зависимости** (например `task_subtype → task_questionary`): для
  каждой категории детерминанта берём одного представителя зависимой колонки (свежайшая
  строка по дате, где значение не NULL) и держим эту связь в синтетике — «опросник
  остаётся у своего подтипа задачи».
- **Чувствительные данные → синтетика**, явно «сгенерированная»: ФИО «Иванов Иван
  Иванович», ИНН «1234567890», компании «ООО Ромашка».

---

## Программа 1 — profiler (закрытый контур)

Подключение задаётся **явно** (в ячейке ноутбука, через `--dsn` или `db_config.json`) —
из проекта ничего не берётся.

```bash
python profiler.py --tables "public.tasks,public.clients" \
    --policy policy.yaml --out profile.json --sample-n 1000000 \
    --dsn "postgresql://user@host:5432/prom"

# либо список таблиц из CSV (колонки schema,table или schema_name,table_name)
python profiler.py --tables-csv tables.csv --policy policy.yaml --out profile.json
```

Порядок разрешения DSN: `--dsn` → `AGC_DB_DSN`/`DB_DSN` → `--db-config db_config.json`
(`host/port/database/user[/password]`). Движок — **read-only**. Пароль можно не
указывать (Kerberos/GSSAPI, как на проде GPDB).

Структуру (типы, nullability, default, storage, ключ распределения, партиции) берём из
системного каталога; PK/статистику/категории/зависимости — из сэмпла в pandas.

### Формат policy-YAML (whitelist)

```yaml
version: 1
columns:
  "public.tasks.task_type":    categorical_keep                 # хранит реальные значения
  "public.tasks.task_subtype": categorical_keep
  "public.clients.name":       {class: pii, generator: full_name}
  "public.tasks.amount":       {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}
tables:
  "public.tasks":
    order_groups: [["created_at", "updated_at"]]                # created <= updated
    dependencies:
      - determinant: task_subtype
        dependent:   task_questionary
        order_by:    task_date                                  # представитель — свежайший
        keep_representative: true                               # реальный представитель (whitelist)
```

Классы: `categorical_keep` | `sensitive_numeric` | `pii` | `key` | `datetime` | `sensitive`.
PK выводится автоматически; `categorical_keep` — только явно (граница безопасности).

---

## Программа 2 — generator (открытый контур)

```bash
python generator.py --profile profile.json --scale 0.001 --seed 42 --format csv --out out/
python generator.py --profile profile.json --format sql --keep-gpdb
```

- `--scale` — доля от исходного числа строк (1.2M × 0.001 ≈ 1200);
- `--keep-gpdb` — сохранить `DISTRIBUTED BY` и заметки о партициях (иначе всё heap);
- таблицы независимы (FK нет); суррогатный PK по гипотезе; категории — все;
  зависимости соблюдены; без ML-синтезаторов (SDV/CTGAN).

---

## Установка (закрытый контур без интернета)

```bash
pip install --no-index --find-links=./wheels -r requirements.txt
```

`SQLAlchemy`, `psycopg2`, `PyYAML`, `pandas` — для profiler. Generator работает на
stdlib; `Faker`/`numpy` — опциональные улучшения.

---

## Структура

```
agc_common.py            общие утилиты (валидация идентификаторов, логирование)
agc_profiler/            ПРОГРАММА 1
  db.py                  подключение (read-only), самостоятельное
  catalog_reader.py      структура из системного каталога (типы, storage, dist, партиции)
  sampler.py             адаптивный сэмпл таблицы в pandas
  analyze.py             PK-гипотеза, статистика, категории, представители зависимостей
  classifier.py          авто-эвристики (только ПРЕДЛАГАЮТ класс)
  policy.py              whitelist чувствительности + зависимости (финальное решение)
  linter.py              строгая проверка утечек реальных значений
  profile_builder.py     сборка profile.json
  cli.py                 CLI
agc_generator/           ПРОГРАММА 2
  profile_parser.py      парсер profile.json
  ddl_builder.py         schema.sql (PK-гипотеза, без FK)
  value_generators.py    генераторы значений по классам (+ зависимости)
  key_linker.py          суррогатный PK, категории, функциональные зависимости
  writer.py              CSV / INSERT-ы
  cli.py                 CLI
profiler.py, generator.py   точки входа
policy.example.yaml, tables.example.csv, examples/profile.example.json
```


In [ ]:
%%writefile generated/policy.example.yaml
# Policy — whitelist чувствительности (граница безопасности).
#
# По умолчанию КАЖДАЯ колонка чувствительна и синтезируется. Реальные значения
# уходят в профиль ТОЛЬКО для колонок, явно помеченных здесь как categorical_keep
# (и для представителей зависимостей с keep_representative — это тоже whitelist).
# Забыли пометить новую колонку с ФИО — она НЕ утечёт (по умолчанию синтезируется).
#
# Классы: categorical_keep | sensitive_numeric | pii | key | datetime | sensitive
# PK выводится гипотезой по сэмплу автоматически; FK не выводится.
version: 1

columns:
  # Бизнес-категории — сохраняем реальные значения и доли (для GROUP BY).
  "public.tasks.task_type":    categorical_keep
  "public.tasks.task_subtype": categorical_keep
  "public.clients.region":     categorical_keep

  # Явно чувствительные — синтетика (это и так дефолт, но фиксируем генератор).
  "public.clients.client_name": {class: pii, generator: full_name}   # Иванов Иван Иванович
  "public.clients.inn":         {class: pii, generator: inn}         # 1234567890
  "public.clients.phone":       {class: pii, generator: phone}

  # Числовые суммы — форма распределения без реальных значений.
  "public.tasks.amount": {class: sensitive_numeric, dist: lognormal, avg_hint: "~5e4"}

tables:
  "public.tasks":
    # Порядок связанных дат: created_at <= updated_at.
    order_groups: [["created_at", "updated_at"]]
    # Функциональная зависимость: для каждого task_subtype берём представителя
    # task_questionary (свежайшая строка по task_date, где он не NULL) и держим
    # эту связь в синтетике. keep_representative=true → реальный представитель
    # (whitelist-opt-in); false → стабильный синтетический токен на подтип.
    dependencies:
      - determinant: task_subtype
        dependent:   task_questionary
        order_by:    task_date
        keep_representative: true


In [ ]:
%%writefile generated/tables.example.csv
schema,table
public,clients
public,tasks
public,payments


In [ ]:
%%writefile generated/examples/profile.example.json
{
  "profile_version": 2,
  "generated_at": "2026-07-20T00:00:00+00:00",
  "source_dialect": "greenplum",
  "note": "Пример профиля v2: реальные значения только в categorical_keep; PK — гипотеза по сэмплу; FK нет.",
  "tables": [
    {
      "schema": "public",
      "table": "clients",
      "relkind": "table",
      "is_view": false,
      "storage": "heap",
      "distributed_by": ["client_id"],
      "partitioned_by": [],
      "row_count": {"value": 500000, "estimated": true, "sample_rows": 500000},
      "pk_hypothesis": ["client_id"],
      "column_dependencies": [],
      "defaults": {},
      "not_null": ["client_id", "client_name"],
      "columns": {
        "client_id":   {"pg_type": "bigint", "policy": "key", "role": "pk", "pk_inferred": true, "n_distinct": 500000, "null_frac": 0.0},
        "client_name": {"pg_type": "text", "policy": "pii", "null_frac": 0.0, "generator": "full_name", "unique_like": false},
        "inn":         {"pg_type": "character varying(12)", "policy": "pii", "null_frac": 0.03, "generator": "inn", "unique_like": true, "char_len": 12},
        "phone":       {"pg_type": "character varying(20)", "policy": "pii", "null_frac": 0.1, "generator": "phone", "unique_like": false, "char_len": 20},
        "region":      {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.01, "n_distinct": 4,
                        "values": [["Центр", 0.4], ["Юг", 0.25], ["Сибирь", 0.2], ["Урал", 0.15]]}
      }
    },
    {
      "schema": "public",
      "table": "tasks",
      "relkind": "table",
      "is_view": false,
      "storage": "append_optimized_column",
      "distributed_by": ["task_id"],
      "partitioned_by": ["task_date"],
      "row_count": {"value": 1200000, "estimated": true, "sample_rows": 1000000},
      "pk_hypothesis": ["task_id"],
      "column_dependencies": [{"determinant": "task_subtype", "dependent": "task_questionary"}],
      "defaults": {},
      "not_null": ["task_id", "task_type"],
      "columns": {
        "task_id":          {"pg_type": "bigint", "policy": "key", "role": "pk", "pk_inferred": true, "n_distinct": 1000000, "null_frac": 0.0},
        "task_type":        {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.0, "n_distinct": 4,
                             "values": [["отток", 0.4], ["привлечение", 0.3], ["сопровождение", 0.2], ["прочее", 0.1]]},
        "task_subtype":     {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.0, "n_distinct": 5,
                             "values": [["A1", 0.35], ["A2", 0.25], ["B1", 0.2], ["B2", 0.15], ["C1", 0.05]]},
        "task_questionary": {"pg_type": "text", "policy": "categorical_keep", "null_frac": 0.05,
                             "depends_on": ["task_subtype"], "n_distinct": 5,
                             "value_map": {"A1": "QST-РАБОТА-С-ОТТОКОМ", "A2": "QST-УДЕРЖАНИЕ", "B1": "QST-ПРИВЛЕЧЕНИЕ", "B2": "QST-КРОСС-ПРОДАЖИ", "C1": "QST-СОПРОВОЖДЕНИЕ"}},
        "status":           {"pg_type": "text", "policy": "categorical_synth", "null_frac": 0.0, "n_distinct": 5,
                             "freqs": [0.5, 0.2, 0.15, 0.1, 0.05]},
        "amount":           {"pg_type": "numeric(14,2)", "policy": "sensitive_numeric", "null_frac": 0.05, "n_distinct": -0.8,
                             "precision": 14, "scale": 2, "dist": "lognormal", "avg_hint": "~5e4"},
        "task_date":        {"pg_type": "date", "policy": "datetime", "null_frac": 0.0,
                             "range": {"start": "2020-01-01", "end": "2024-12-31"}},
        "created_at":       {"pg_type": "timestamp without time zone", "policy": "datetime", "null_frac": 0.0,
                             "range": {"start": "2020-01-01", "end": "2024-12-31"}, "order_group": true},
        "updated_at":       {"pg_type": "timestamp without time zone", "policy": "datetime", "null_frac": 0.1,
                             "range": {"start": "2020-01-01", "end": "2024-12-31"}, "order_group": true}
      }
    }
  ]
}


## 3. profiler — закрытый контур

Подключаемся по заданному выше DSN (самостоятельно, без проекта). Структуру берём из каталога, а PK-гипотезу / статистику / категории / зависимости — из сэмпла в pandas. Если живой БД нет — берём готовый пример профиля, чтобы продемонстрировать generator. Движок read-only; реальные строки наружу не выгружаются.

In [ ]:
import sys, json
sys.path.insert(0, str(GEN))                         # agc_* модули

from agc_profiler.cli import parse_tables_arg, parse_tables_csv
from agc_profiler.policy import Policy
from agc_profiler.profile_builder import build_profile
from agc_profiler.linter import check_profile
from agc_profiler.db import make_engine, dsn_from_parts

tables = parse_tables_csv(TABLES_CSV) if TABLES_CSV else parse_tables_arg(TABLES)
policy = Policy.load(POLICY_YAML or None)
profile_path = OUT / "profile.json"

dsn = DB_DSN or dsn_from_parts(DB_HOST, DB_NAME, port=DB_PORT, user=DB_USER, password=DB_PASSWORD)
try:
    engine = make_engine(dsn)
    engine.connect().close()
    profile = build_profile(engine, tables, policy, sample_n=SAMPLE_N)
    check_profile(profile)                           # строгий линтер утечек
    profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"profiler отработал вживую → {profile_path}")
except Exception as e:
    print(f"[i] Живое подключение к БД недоступно ({type(e).__name__}: {e}).")
    print("    Использую готовый пример профиля для демонстрации генератора.")
    import shutil
    shutil.copy(GEN / "examples" / "profile.example.json", profile_path)
    print(f"    Профиль: {profile_path}")

## 4. Линтер профиля — аудит утечек

Главная точка аудита: если колонка **не** `categorical_keep`, но в её профиле есть реальные значения — падаем. Здесь же — сводка классов по таблицам.

In [ ]:
prof = json.loads(profile_path.read_text(encoding="utf-8"))
check_profile(prof)
print("Линтер OK: реальные значения только в categorical_keep.\n")
for t in prof["tables"]:
    kinds = {}
    for c in t["columns"].values():
        kinds[c["policy"]] = kinds.get(c["policy"], 0) + 1
    rc = t["row_count"]["value"]
    print(f"  {t['schema']}.{t['table']}: rows~{rc}  classes={kinds}")

## 5. generator — открытый контур

`schema.sql` (DDL, PK-гипотеза без FK) + синтетические данные. Категории сохраняются все, чувствительные поля синтезируются, функциональные зависимости (`task_subtype → task_questionary`) соблюдены.

In [ ]:
from agc_generator.profile_parser import load_profile
from agc_generator.ddl_builder import build_ddl
from agc_generator.key_linker import generate
from agc_generator.writer import write_csv, write_sql

gprof = load_profile(profile_path)
(OUT / "schema.sql").write_text(build_ddl(gprof, keep_gpdb=KEEP_GPDB), encoding="utf-8")
data = generate(gprof, SCALE_FACTOR, SEED)
if OUTPUT_FORMAT == "csv":
    write_csv(gprof, data, OUT / "data")
else:
    write_sql(gprof, data, OUT / "data.sql")

total = sum(len(v) for v in data.values())
print(f"\nГотово: {len(data)} таблиц, {total} строк суммарно → {OUT}")

## 6. Предпросмотр результата

In [ ]:
print("=== generated/ (инструмент) ===")
for p in sorted(GEN.rglob("*")):
    if p.is_file() and "output" not in p.parts:
        print("  ", p.relative_to(NB_DIR))
print("\n=== output/ (артефакты запуска) ===")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(NB_DIR), f"{p.stat().st_size} B")
print("\n--- schema.sql (первые строки) ---")
print("\n".join((OUT / "schema.sql").read_text(encoding="utf-8").splitlines()[:22]))

## Запуск как отдельных CLI (для реального переноса между контурами)

```bash
# закрытый контур:
python generated/profiler.py --tables "public.tasks,public.clients" \
    --policy generated/policy.example.yaml --out profile.json \
    --dsn "postgresql://user@host:5432/prom" --sample-n 1000000

# перенести profile.json через зазор, затем на открытом контуре:
python generated/generator.py --profile profile.json --scale 0.001 \
    --seed 42 --format csv --out out/
```

Подробности — в `generated/README.md`.